# **LSTM ** **MultiClass Classification**

              EMOTION DATASET
                    ↓
             Text + Labels
                    ↓
             Text Cleaning
                    ↓
              Tokenization
                    ↓
              Vocabulary
                    ↓
              Word → IDs
                    ↓
                Padding
                    ↓
                Tensors
                    ↓
               DataLoader
                    ↓
               Embedding
                    ↓
                  LSTM
                    ↓
             Hidden State
                    ↓
              Linear Layer
                    ↓
             6 Class Scores
                    ↓
          CrossEntropyLoss
                    ↓
             Backpropagation
                    ↓
              Adam Optimizer
                    ↓
              Emotion 🎯

# **Import the Libraries**

In [ ]:
import pandas as pd
import numpy as np

import torch
import torch.nn as nn
import torch.optim as optim

from torch.utils.data import TensorDataset, DataLoader

from sklearn.model_selection import train_test_split
from collections import Counter

In [ ]:
from datasets import load_dataset

dataset = load_dataset("dair-ai/emotion")

print(dataset)

README.md:   0%|          | 0.00/9.05k [00:00<?, ?B/s]

split/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 1.03MB            

split/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

split/validation-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  127kB            

split/validation-00000-of-00001.parquet: downloading bytes:           |  0.00B            

...

split/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/16000 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/2000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/2000 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 16000
    })
    validation: Dataset({
        features: ['text', 'label'],
        num_rows: 2000
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 2000
    })
})


In [ ]:
print("Training:", len(dataset["train"]))
print("Validation:", len(dataset["validation"]))
print("Test:", len(dataset["test"]))

Training: 16000
Validation: 2000
Test: 2000


**First: Separate Text and Labels**

In [ ]:
train_texts = dataset["train"]["text"]
train_labels = dataset["train"]["label"]

val_texts = dataset["validation"]["text"]
val_labels = dataset["validation"]["label"]

test_texts = dataset["test"]["text"]
test_labels = dataset["test"]["label"]

**Convert Text to Lowercase**

In [ ]:
train_texts = [text.lower() for text in train_texts]
val_texts = [text.lower() for text in val_texts]
test_texts = [text.lower() for text in test_texts]

**Tokenization**

In [ ]:
train_tokens = [text.split() for text in train_texts]
val_tokens = [text.split() for text in val_texts]
test_tokens = [text.split() for text in test_texts]

In [ ]:
print(train_texts[0])
print(train_tokens[0])
print(train_labels[0])

i didnt feel humiliated
['i', 'didnt', 'feel', 'humiliated']
0


**Building Vocabulary**

In [ ]:
word_counter = Counter()

for sentence in train_tokens:
    word_counter.update(sentence)

vocab = {
    "<PAD>": 0,
    "<UNK>": 1
}

for word in word_counter:
    vocab[word] = len(vocab)

print("Vocabulary Size:", len(vocab))

Vocabulary Size: 15214


**Convert Words → IDs**

In [12]:
def text_to_ids(tokens, vocab):

    ids = []

    for word in tokens:

        if word in vocab:
            ids.append(vocab[word])
        else:
            ids.append(vocab["<UNK>"])

    return ids

**Convert the Entire Dataset**

In [13]:
train_ids = [
    text_to_ids(sentence, vocab)
    for sentence in train_tokens
]

val_ids = [
    text_to_ids(sentence, vocab)
    for sentence in val_tokens
]

test_ids = [
    text_to_ids(sentence, vocab)
    for sentence in test_tokens
]

print(train_tokens[0])
print(train_ids[0])
print(train_labels[0])

['i', 'didnt', 'feel', 'humiliated']
[2, 3, 4, 5]
0


**Finding the Maximum Sequence Length**

In [14]:
max_length = max(
    len(sentence)
    for sentence in train_ids
)

print("Maximum Sequence Length:", max_length)

Maximum Sequence Length: 66


**Create the Padding Function**

In [15]:
def pad_sequence(sequence, max_length):

    if len(sequence) < max_length:

        return sequence + [0] * (max_length - len(sequence))

    else:

        return sequence[:max_length]

**Apply Padding**

In [17]:
train_padded = [
    pad_sequence(sentence, max_length)
    for sentence in train_ids
]

val_padded = [
    pad_sequence(sentence, max_length)
    for sentence in val_ids
]

test_padded = [
    pad_sequence(sentence, max_length)
    for sentence in test_ids
]

print(train_padded[0])
print(len(train_padded[0]))

[2, 3, 4, 5, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
66


# Convert to Pytorch Tensors

In [19]:
# Training dataset
X_train = torch.tensor(
    train_padded,
    dtype=torch.long
)

y_train = torch.tensor(
    train_labels,
    dtype=torch.long
)

# validation dataset
X_val = torch.tensor(
    val_padded,
    dtype=torch.long
)

y_val = torch.tensor(
    val_labels,
    dtype=torch.long
)

# Testing dataset
X_test = torch.tensor(
    test_padded,
    dtype=torch.long
)

y_test = torch.tensor(
    test_labels,
    dtype=torch.long
)

print("X_train:", X_train.shape)
print("y_train:", y_train.shape)

print("X_val:", X_val.shape)
print("y_val:", y_val.shape)

print("X_test:", X_test.shape)
print("y_test:", y_test.shape)

X_train: torch.Size([16000, 66])
y_train: torch.Size([16000])
X_val: torch.Size([2000, 66])
y_val: torch.Size([2000])
X_test: torch.Size([2000, 66])
y_test: torch.Size([2000])


**Create TensorDataset**

In [20]:
train_dataset = TensorDataset(
    X_train,
    y_train
)

val_dataset = TensorDataset(
    X_val,
    y_val
)

test_dataset = TensorDataset(
    X_test,
    y_test
)

Create **DataLoader**

In [21]:
batch_size = 32

train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=batch_size,
    shuffle=False
)

test_loader = DataLoader(
    test_dataset,
    batch_size=batch_size,
    shuffle=False
)

# **LSTM Cell Architecture**

                         xₜ
                         │
                         │
                         ↓
                  ┌─────────────┐
                  │    LSTM     │
                  │             │
        hₜ₋₁ ────→│   Gates     │
                  │             │
                  └──────┬──────┘
                         │
             ┌───────────┴───────────┐
             ↓                       ↓
        Forget Gate             Input Gate
             ↓                       ↓
       Old Memory             Candidate Memory
             │                       │
             └───────────┬───────────┘
                         ↓
                  New Cell State
                         │
                         ↓
                  Output Gate
                         │
                         ↓
                   Hidden State
                         │
                 ┌───────┴────────┐
                 ↓                ↓
             Next LSTM        Classifier
               step

In [22]:
class EmotionLSTM(nn.Module):

    def __init__(
        self,
        vocab_size,
        embedding_dim,
        hidden_dim,
        num_classes
    ):
        super().__init__()

        self.embedding = nn.Embedding(
            num_embeddings=vocab_size,
            embedding_dim=embedding_dim,
            padding_idx=0
        )

        self.lstm = nn.LSTM(
            input_size=embedding_dim,
            hidden_size=hidden_dim,
            batch_first=True
        )

        self.fc = nn.Linear(
            hidden_dim,
            num_classes
        )

    def forward(self, x):

      embedded = self.embedding(x)

      output, (hidden, cell) = self.lstm(embedded)

      hidden = hidden[-1]

      logits = self.fc(hidden)

      return logits

**Create the Model**

In [24]:
vocab_size = len(vocab)
embedding_dim = 100
hidden_dim = 128
num_classes = 6

model = EmotionLSTM(
    vocab_size,
    embedding_dim,
    hidden_dim,
    num_classes
)

print(model)

EmotionLSTM(
  (embedding): Embedding(15214, 100, padding_idx=0)
  (lstm): LSTM(100, 128, batch_first=True)
  (fc): Linear(in_features=128, out_features=6, bias=True)
)


# **Training Loop**

In [33]:
criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.001
)

epochs = 10

for epoch in range(epochs):

    model.train()

    total_loss = 0
    correct = 0
    total = 0

    for X_batch, y_batch in train_loader:

        # Forward
        logits = model(X_batch)

        # Loss
        loss = criterion(logits, y_batch)

        # Clear gradients
        optimizer.zero_grad()

        # Backpropagation
        loss.backward()

        # Update weights
        optimizer.step()

        total_loss += loss.item()

        # Training accuracy
        predictions = torch.argmax(logits, dim=1)

        correct += (predictions == y_batch).sum().item()
        total += y_batch.size(0)

    train_accuracy = correct / total

    # Validation
    val_accuracy = evaluate(
        model,
        val_loader
    )

    avg_loss = total_loss / len(train_loader)

    print(
        f"Epoch {epoch+1}/{epochs} | "
        f"Loss: {avg_loss:.4f} | "
        f"Train Acc: {train_accuracy*100:.2f}% | "
        f"Val Acc: {val_accuracy*100:.2f}%"
    )

Accuracy: 57.70%
Epoch 1/10 | Loss: 0.7665 | Train Acc: 61.79% | Val Acc: 57.70%
Accuracy: 62.50%
Epoch 2/10 | Loss: 0.7082 | Train Acc: 63.21% | Val Acc: 62.50%
Accuracy: 66.25%
Epoch 3/10 | Loss: 0.6517 | Train Acc: 67.01% | Val Acc: 66.25%
Accuracy: 76.80%
Epoch 4/10 | Loss: 0.5987 | Train Acc: 72.56% | Val Acc: 76.80%
Accuracy: 85.70%
Epoch 5/10 | Loss: 0.3991 | Train Acc: 87.33% | Val Acc: 85.70%
Accuracy: 87.20%
Epoch 6/10 | Loss: 0.2953 | Train Acc: 91.54% | Val Acc: 87.20%
Accuracy: 87.95%
Epoch 7/10 | Loss: 0.2309 | Train Acc: 93.53% | Val Acc: 87.95%
Accuracy: 87.60%
Epoch 8/10 | Loss: 0.2013 | Train Acc: 94.47% | Val Acc: 87.60%
Accuracy: 88.35%
Epoch 9/10 | Loss: 0.1874 | Train Acc: 94.82% | Val Acc: 88.35%
Accuracy: 88.45%
Epoch 10/10 | Loss: 0.1498 | Train Acc: 95.71% | Val Acc: 88.45%


# **Validation**

In [34]:
def evaluate(model, data_loader):

    model.eval()

    correct = 0
    total = 0

    with torch.no_grad():

        for X_batch, y_batch in data_loader:

            logits = model(X_batch)

            predictions = torch.argmax(logits, dim=1)

            correct += (predictions == y_batch).sum().item()

            total += y_batch.size(0)

    accuracy = correct / total

    print(f"Accuracy: {accuracy * 100:.2f}%")

    return accuracy


# IMPORTANT: Call the function
accuracy = evaluate(model, val_loader)

Accuracy: 88.45%
